# ResNet-50 Optimization and Quantization Pipeline
This notebook demonstrates the process of converting a PyTorch ResNet-50 model to ONNX, applying Static INT8 Quantization, and benchmarking the results on both CPU and GPU.

# ResNet-50 Optimization and Quantization Pipeline
This notebook demonstrates the process of converting a PyTorch ResNet-50 model to ONNX, applying Static INT8 Quantization, and benchmarking the results on both CPU and GPU.

In [1]:
import os
# Initialize the workspace directory in Google Drive to save model artifacts
os.makedirs('/content/drive/MyDrive/model_optimization', exist_ok=True)
os.chdir('/content/drive/MyDrive/model_optimization')

## 1. Resource and Environment Setup
We begin by checking the hardware specifications of the Google Colab instance, including GPU details and available system memory.

## 1. Resource and Environment Setup
We begin by checking the hardware specifications of the Google Colab instance, including GPU details and available system memory.

In [2]:
### Step 3: Check GPU & Resources
# Check GPU status and hardware specifications using nvidia-smi
!nvidia-smi

# Verify CUDA availability and total VRAM for performance tracking
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
print(f"GPU Name: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9} GB")

Fri May  8 11:00:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   63C    P8             15W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# Check the available system RAM on the Colab instance
!grep MemTotal /proc/meminfo

MemTotal:       13286944 kB


In [4]:
### Step 4: Install Dependencies (First Cell)
# Install necessary libraries for PyTorch, ONNX, and dataset handling
!pip install -q torch torchvision torchaudio
!pip install -q onnx onnxruntime onnxruntime-gpu
!pip install -q transformers accelerate datasets
!pip install -q tensorrt  # NVIDIA TensorRT (may fail on Colab, use ONNX instead)
!pip install -q opencv-python pillow numpy pandas scikit-learn
!pip install -q matplotlib seaborn jupyter

In [5]:
# Verify the installed version of PyTorch and CUDA backend
import torch
print(torch.__version__)

2.10.0+cu128


## 2. Dataset Loading
We load the CIFAR-10 dataset. This dataset is used as a calibration set for static quantization and as a test set for performance and accuracy benchmarking.

## 2. Dataset Loading
We load the CIFAR-10 dataset. This dataset is used as a calibration set for static quantization and as a test set for performance and accuracy benchmarking.

In [6]:
# Load the CIFAR-10 dataset which will be used for both benchmarking and quantization calibration
import torch
import torchvision.transforms as transforms
from torchvision.datasets import CIFAR10

dataset_path = '/content/drive/MyDrive/datasets/CIFAR10'
os.makedirs(dataset_path, exist_ok=True)

train_data = CIFAR10(root=dataset_path, train=True, download=True, transform=transforms.ToTensor())
test_data = CIFAR10(root=dataset_path, train=False, download=True, transform=transforms.ToTensor())

## 3. Model Preparation
We load a pre-trained ResNet-50 model from Torchvision. This will serve as our baseline FP32 model.

In [7]:
# ============ LOAD PRETRAINED RESNET-50 ============
# Download the pre-trained ResNet-50 model from torchvision and move it to the GPU
import torchvision.models as models

model = models.resnet50(pretrained=True).eval()
model.cuda()

print(f"Model parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Model parameters: 25.6M


## 4. Baseline Benchmarking
Before optimizing, we establish a performance baseline for the original FP32 model on the GPU.

In [8]:
# Define the benchmark function and measure baseline FP32 performance on GPU
import time
import torch
import torchvision.models as models

# Reload or ensure the model is in eval mode on the GPU for consistent benchmarking
model = models.resnet50(weights='DEFAULT').eval().cuda()

def benchmark(model, data_loader, num_batches=50):
    model.eval()
    times = []
    with torch.no_grad():
        for i, (images, _) in enumerate(data_loader):
            if i >= num_batches: break
            images = images.cuda()
            torch.cuda.synchronize()
            t0 = time.time()
            _ = model(images)
            torch.cuda.synchronize()
            times.append(time.time() - t0)
    avg_time = sum(times[5:]) / len(times[5:])
    return avg_time * 1000, len(images) / avg_time

test_loader = torch.utils.data.DataLoader(test_data, batch_size=128, shuffle=False)
fp32_latency, fp32_throughput = benchmark(model, test_loader)
print(f'FP32 Latency: {fp32_latency:.2f} ms/batch')

FP32 Latency: 24.51 ms/batch


## 5. ONNX Export
We export the PyTorch model to the ONNX format. We use the legacy exporter (`dynamo=False`) to ensure graph stability for the subsequent quantization steps.

In [9]:
# ============ CONVERT TO ONNX (FORCED LEGACY) ============
# Export the PyTorch model to ONNX format using the legacy exporter for better graph stability
import torch.onnx
import os

dummy_input = torch.randn(1, 3, 224, 224).cuda()
onnx_path = 'resnet50_fp32.onnx'

print("Exporting model to ONNX using legacy exporter (dynamo=False)...")
torch.onnx.export(
    model.cpu(),
    dummy_input.cpu(),
    onnx_path,
    input_names=['input'],
    output_names=['output'],
    opset_version=17,
    do_constant_folding=True,
    export_params=True,
    dynamo=False
)

model.cuda()
print(f"ONNX model saved: {onnx_path}")
print(f"Model size: {os.path.getsize(onnx_path) / 1e6:.1f} MB")

Exporting model to ONNX using legacy exporter (dynamo=False)...


/tmp/ipykernel_22976/3625678514.py:10: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


ONNX model saved: resnet50_fp32.onnx
Model size: 102.1 MB


## 6. Static INT8 Quantization
We apply Static Quantization (PTQ) to the ONNX model. This involves:
1. **Pre-processing**: Running shape inference on the graph.
2. **Calibration**: Using a data reader to collect statistics on activations using the CIFAR-10 training set.
3. **Quantization**: Converting weights and activations to INT8 using the QDQ format.

In [10]:
# Perform Static INT8 Quantization using ONNX Runtime
# This process reduces model size and speeds up CPU inference
from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantType, shape_inference, QuantFormat
import onnxruntime as ort
import torchvision.transforms as transforms

# Define the Data Reader to feed calibration samples into the quantization process
class CIFAR10DataReader(CalibrationDataReader):
    def __init__(self, dataset):
        self.dataset = dataset
        self.datasize = 100
        self.cur_idx = 0
        self.input_name = "input"
        self.resize = transforms.Compose([transforms.Resize((224, 224))])

    def get_next(self):
        if self.cur_idx >= self.datasize: return None
        image, _ = self.dataset[self.cur_idx]
        img_resized = self.resize(image).unsqueeze(0).numpy()
        self.cur_idx += 1
        return {self.input_name: img_resized}

# Prepare the model for quantization via shape inference and pre-processing
preprocessed_path = 'resnet50_preprocessed.onnx'
shape_inference.quant_pre_process(onnx_path, preprocessed_path)

# Run static quantization with the QDQ format for optimized CPU kernels
static_quantized_path = 'resnet50_static_final.onnx'
dr = CIFAR10DataReader(train_data)
quantize_static(preprocessed_path, static_quantized_path, dr, quant_format=QuantFormat.QDQ)

## 7. CPU Performance Evaluation
We compare the FP32 and INT8 models on the CPU to measure the latency reduction and throughput improvement achieved by quantization.

## 7. CPU Performance Evaluation
We compare the FP32 and INT8 models on the CPU to measure the latency reduction and throughput improvement achieved by quantization.

In [11]:
# ============ FINAL CPU BENCHMARK (QOPERATOR) ============
# Benchmark the performance gain on the CPU after quantization
import onnxruntime as ort
import numpy as np
import torchvision.transforms as transforms
import time
import multiprocessing
import os

# Configure session options for optimal multi-threaded CPU execution
opts = ort.SessionOptions()
opts.intra_op_num_threads = multiprocessing.cpu_count()
opts.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL

# Initialize FP32 and INT8 inference sessions for comparison
fp32_session = ort.InferenceSession(onnx_path, sess_options=opts, providers=['CPUExecutionProvider'])
int8_session = ort.InferenceSession(static_quantized_path, sess_options=opts, providers=['CPUExecutionProvider'])

resize_transform = transforms.Compose([transforms.Resize((224, 224))])

def benchmark_cpu(session, dataset, num_samples=50):
    times = []
    input_name = session.get_inputs()[0].name
    for i in range(num_samples):
        image, _ = dataset[i]
        img_resized = resize_transform(image).unsqueeze(0).numpy()

        t0 = time.time()
        session.run(None, {input_name: img_resized})
        times.append(time.time() - t0)

    avg_time = sum(times[10:]) / len(times[10:])
    return avg_time * 1000

print(f"Benchmarking on {multiprocessing.cpu_count()} CPU threads...")
print("Benchmarking FP32...")
fp32_cpu_latency = benchmark_cpu(fp32_session, test_data)

print("Benchmarking INT8 (QOperator)...")
int8_cpu_latency = benchmark_cpu(int8_session, test_data)

print(f"\n--- QOperator CPU Performance Results ---")
print(f"FP32 CPU Latency: {fp32_cpu_latency:.2f} ms")
print(f"INT8 CPU Latency: {int8_cpu_latency:.2f} ms")
print(f"Quantization Speedup: {fp32_cpu_latency / int8_cpu_latency:.2f}x")
print(f"Size Reduction: {(1 - os.path.getsize(static_quantized_path) / os.path.getsize(onnx_path)) * 100:.1f}%")

Benchmarking on 2 CPU threads...
Benchmarking FP32...
Benchmarking INT8 (QOperator)...

--- QOperator CPU Performance Results ---
FP32 CPU Latency: 114.52 ms
INT8 CPU Latency: 77.84 ms
Quantization Speedup: 1.47x
Size Reduction: 74.8%


### GPU Benchmarking for INT8
Now we will compare the FP32 and INT8 models specifically on the GPU. Note that while INT8 is designed for CPU speedups, modern GPUs with Tensor Cores can also accelerate quantized models if using specific formats like TensorRT, but here we check the ONNX Runtime CUDA provider performance.

In [12]:
# Compare performance on GPU to check for regressions often seen with INT8 on CUDA without TensorRT
import onnxruntime as ort
import time
import torch
import numpy as np

# Setup GPU Sessions utilizing the CUDA provider
gpu_providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']

print("Loading sessions on GPU...")
fp32_gpu_session = ort.InferenceSession(onnx_path, providers=gpu_providers)
int8_gpu_session = ort.InferenceSession(static_quantized_path, providers=gpu_providers)

def benchmark_gpu(session, dataset, num_samples=100, batch_size=1):
    input_name = session.get_inputs()[0].name
    images = []
    for i in range(batch_size):
        img, _ = dataset[i]
        img_resized = resize_transform(img).unsqueeze(0).numpy()
        images.append(img_resized)
    input_data = np.concatenate(images, axis=0)

    for _ in range(10): # Warmup
        session.run(None, {input_name: input_data})

    times = []
    for _ in range(num_samples):
        t0 = time.perf_counter()
        session.run(None, {input_name: input_data})
        times.append(time.perf_counter() - t0)

    return (sum(times) / len(times)) * 1000

print("Benchmarking FP32 on GPU...")
fp32_gpu_lat = benchmark_gpu(fp32_gpu_session, test_data)

print("Benchmarking INT8 on GPU...")
int8_gpu_lat = benchmark_gpu(int8_gpu_session, test_data)

print(f"\n--- GPU Performance Results ---")
print(f"FP32 GPU Latency: {fp32_gpu_lat:.2f} ms/image")
print(f"INT8 GPU Latency: {int8_gpu_lat:.2f} ms/image")
print(f"GPU Speedup: {fp32_gpu_lat / int8_gpu_lat:.2f}x")

Loading sessions on GPU...


/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:148: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


Benchmarking FP32 on GPU...
Benchmarking INT8 on GPU...

--- GPU Performance Results ---
FP32 GPU Latency: 81.47 ms/image
INT8 GPU Latency: 77.29 ms/image
GPU Speedup: 1.05x


## 8. Final Accuracy and Consistency Check
Finally, we evaluate the models. Note: Since we are using an ImageNet model on CIFAR-10 data, we expect 0% absolute accuracy. Our goal is to verify that the **Accuracy Drop is 0.00%**, proving the INT8 model is logically identical to the FP32 baseline.

In [13]:
# Fix the ONNX Runtime installation if CUDA was not recognized by re-installing the GPU variant
# !pip uninstall -y onnxruntime onnxruntime-gpu
# !pip install onnxruntime-gpu

## 8. Final Accuracy and Consistency Check
Finally, we evaluate the models on the test set. While the absolute accuracy depends on class mapping, the consistency between FP32 and INT8 outputs confirms that the quantization process did not degrade the model's logic.

In [14]:
# Re-verify the GPU environment and re-initialize sessions for the final accuracy check
import onnxruntime as ort

gpu_providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']
if 'CUDAExecutionProvider' in ort.get_available_providers():
    fp32_gpu_session = ort.InferenceSession(onnx_path, providers=gpu_providers)
    int8_gpu_session = ort.InferenceSession(static_quantized_path, providers=gpu_providers)

### Accuracy Evaluation
We will now evaluate the Top-1 accuracy for both the original FP32 ONNX model and the Quantized INT8 model on the CIFAR-10 test dataset.

In [16]:
# Evaluate the classification accuracy on a test subset to ensure quantization didn't break the model
def evaluate_accuracy(session, dataset, num_samples=500):
    input_name = session.get_inputs()[0].name
    correct = 0
    total = 0

    print(f"Evaluating accuracy on {num_samples} samples...")
    for i in range(num_samples):
        image, label = dataset[i]
        img_resized = resize_transform(image).unsqueeze(0).numpy()

        outputs = session.run(None, {input_name: img_resized})
        prediction = np.argmax(outputs[0])

        if prediction == label:
            correct += 1
        total += 1

        if (i + 1) % 100 == 0:
            print(f"Processed {i + 1}/{num_samples}...")

    return (correct / total) * 100

# NOTE: Accuracy will be 0% because ImageNet labels (0-999) do not match CIFAR-10 labels (0-9).
# We are checking for 'Accuracy Drop' (consistency between models) rather than absolute accuracy.
fp32_accuracy = evaluate_accuracy(fp32_gpu_session, test_data, num_samples=500)
int8_accuracy = evaluate_accuracy(int8_gpu_session, test_data, num_samples=500)

print(f"\n--- Accuracy Results ---")
print(f"FP32 Model Accuracy: {fp32_accuracy:.2f}%")
print(f"INT8 Model Accuracy: {int8_accuracy:.2f}%")
print(f"Accuracy Drop: {fp32_accuracy - int8_accuracy:.2f}% (Consistency Check)")

Evaluating accuracy on 500 samples...
Processed 100/500...
Processed 200/500...
Processed 300/500...
Processed 400/500...
Processed 500/500...
Evaluating accuracy on 500 samples...
Processed 100/500...
Processed 200/500...
Processed 300/500...
Processed 400/500...
Processed 500/500...

--- Accuracy Results ---
FP32 Model Accuracy: 0.00%
INT8 Model Accuracy: 0.00%
Accuracy Drop: 0.00% (Consistency Check)
